Data Collection and Label Verification - Foysal

This notebook inspects the two original datasets used in our clickbait
classification project.

The task is to predict whether a news headline is clickbait (1) or
non-clickbait (0)

For train1.csv, the headline column contains the input and the clickbait
column contains the existing numeric label

For train2.csv, the title column contains the input. Its existing labels are mapped as clickbait = 1 and news = 0

This inspection checks dataset sizes, label counts, missing values,
duplicate headlines and conflicting labels

The results are saved in data_report.txt and duplicate_review.csv

In [15]:
from google.colab import files
from pathlib import Path

files.upload()

data_dir = Path("/content")
output_dir = Path("/content/foysal_results")

Saving train1.csv to train1 (4).csv
Saving train2.csv to train2 (3).csv


In [16]:
import csv
from collections import Counter, defaultdict
from pathlib import Path
import unicodedata

# 1 means clickbait and 0 means non-clickbait
settings = {
    "train1.csv": ("headline", "clickbait", {"0": 0, "1": 1}),
    "train2.csv": ("title", "label", {"news": 0, "clickbait": 1}),
}

#keeps the matching headlines together to find duplicates
groups = defaultdict(list)
combined_labels = Counter() #to count the levels for both dataset
total_rows = 0

#text report
report_lines = [
    "DATA INSPECTION - FOYSAL",
    "",
    "Task: classify headlines as clickbait (1) or non-clickbait (0).",
    "",
]

#to load each dataset after checking that its CSV fike has the required text and label columns
for filename, (text_column, label_column, mapping) in settings.items():
    with (data_dir / filename).open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        columns = reader.fieldnames or []
        if not {text_column, label_column}.issubset(columns):
            raise ValueError(f"Unexpected columns in {filename}: {columns}")
        rows = list(reader)

    #counts for the dataasets
    label_counts = Counter()
    headline_counts = Counter()
    missing_headlines = 0
    missing_labels = 0
    unexpected_labels = 0

    #checks that each CSV row has the same number of fields as the header
    for record_number, row in enumerate(rows, start=1):
        if None in row or any(value is None for value in row.values()):
            raise ValueError(f"Malformed CSV record {record_number} in {filename}")

        headline = row[text_column]
        original_label = row[label_column]
        label = mapping.get(original_label) #converts the label to 0 or 1
        label_counts[original_label] += 1
        missing_headlines += int(not headline.strip())
        missing_labels += int(not original_label.strip())
        unexpected_labels += int(bool(original_label.strip()) and label is None)

        if label is not None:
            combined_labels[label] += 1

        #this ignores the differences in capitalization, unicode and spacing when comparing headlines, while keep the original text
        normalized = unicodedata.normalize("NFKC", headline).casefold()
        key = " ".join(normalized.split())

        if key:  #to store each nonempty headline for the duplicate check
            headline_counts[key] += 1
            groups[key].append({
                "source_file": filename,
                "record_number": record_number,
                "headline": headline,
                "original_label": original_label,
                "label": label,
            })

    #for counting extra copies of headlines within the dataset
    repeated_rows = sum(count - 1 for count in headline_counts.values())
    total_rows += len(rows)
    report_lines.extend([  #adding dataset's results to the report
        filename,
        f"Rows: {len(rows):,}",
        f"Columns: {', '.join(columns)}",
        f"Original label counts: {dict(label_counts)}",
        f"Label mapping: {mapping}",
        f"Missing headlines: {missing_headlines}",
        f"Missing labels: {missing_labels}",
        f"Unexpected labels: {unexpected_labels}",
        f"Repeated normalized headline rows beyond the first: {repeated_rows}",
        "",
    ])

#preparing the duplicate review result
duplicate_rows = []
duplicate_groups = 0
cross_source_groups = 0
conflicting_groups = 0

for key, records in sorted(groups.items()):
    if len(records) < 2:
        continue

    duplicate_groups += 1
    cross_source = len({row["source_file"] for row in records}) > 1
    labels = {row["label"] for row in records if row["label"] is not None}
    conflicting_labels = len(labels) > 1
    cross_source_groups += int(cross_source)
    conflicting_groups += int(conflicting_labels)

    for record in records: #gives matching rows the same group number
        duplicate_rows.append({
            **record,
            "group_id": duplicate_groups,
            "cross_source": cross_source,
            "conflicting_labels": conflicting_labels,
        })

#to create the output folder and save duplicate rows to review
output_dir.mkdir(parents=True, exist_ok=True)
with (output_dir / "duplicate_review.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(file, fieldnames=[
        "source_file", "record_number", "headline", "original_label",
        "label", "group_id", "cross_source", "conflicting_labels",
    ])
    writer.writeheader()
    writer.writerows(duplicate_rows)

#adding the combined results and cleaning notes for the report
report_lines.extend([
    f"Combined rows: {total_rows:,}",
    f"Clickbait headlines (1): {combined_labels[1]:,}",
    f"Non-clickbait headlines (0): {combined_labels[0]:,}",
    f"Duplicate headline groups: {duplicate_groups}",
    f"Groups appearing in both datasets: {cross_source_groups}",
    f"Groups with conflicting labels: {conflicting_groups}",
    "",
    "INSPECTION NOTES",
    "Original source labels were retained. No new manual labels were created.",
    "train2 contains title and label only",
    "Duplicate checks normalize Unicode, capitalization and spacing.",
    "Similar headlines with different wording are not detected by this check.",
    "record_number counts data records starting at 1, excluding the header.",
    "Rows with the same group_id belong to the same duplicate headline group.",
    "",
    "HANDOFF TO MOHAMMED",
    "Review duplicate_review.csv and document how conflicting labels are handled.",
    "Prevent duplicate headlines from appearing in different dataset splits.",
    "Provide the same final splits to the baseline and BERT models.",
    "",
    "DATASET SOURCE",
    "https://www.kaggle.com/datasets/vikassingh1996/news-clickbait-dataset",
])

#saving the report as a text file
report_text = "\n".join(report_lines) + "\n"
(output_dir / "data_report.txt").write_text(report_text, encoding="utf-8")

print(report_text)
print("Saved data_report.txt and duplicate_review.csv.")

DATA INSPECTION - FOYSAL

Task: classify headlines as clickbait (1) or non-clickbait (0).

train1.csv
Rows: 32,000
Columns: headline, clickbait
Original label counts: {'1': 15999, '0': 16001}
Label mapping: {'0': 0, '1': 1}
Missing headlines: 0
Missing labels: 0
Unexpected labels: 0
Repeated normalized headline rows beyond the first: 1

train2.csv
Rows: 21,029
Columns: label, title
Original label counts: {'news': 16738, 'clickbait': 4291}
Label mapping: {'news': 0, 'clickbait': 1}
Missing headlines: 0
Missing labels: 0
Unexpected labels: 0
Repeated normalized headline rows beyond the first: 142

Combined rows: 53,029
Clickbait headlines (1): 20,290
Non-clickbait headlines (0): 32,739
Duplicate headline groups: 87
Groups appearing in both datasets: 2
Groups with conflicting labels: 25

INSPECTION NOTES
Original source labels were retained. No new manual labels were created.
train2 contains title and label only
Duplicate checks normalize Unicode, capitalization and spacing.
Similar headl

In [17]:
files.download(str(output_dir / "data_report.txt"))
files.download(str(output_dir / "duplicate_review.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>